In [7]:
from jaxtyping import Integer, Float, Array
from flax.struct import PyTreeNode
import jax.numpy as jnp
from plum import dispatch, Val
from jax import lax

In [8]:
class LinearAssociativeMemory(PyTreeNode):
    pass

class LinearAssociativeMfc(LinearAssociativeMemory):
    state: Float[Array, "item_features context_features"]

class LinearAssociativeMcf(LinearAssociativeMemory):
    state: Float[Array, """context_features item_features"""]

In [9]:
@dispatch
def initialize(
    model_type: Val(LinearAssociativeMfc),
    item_count: int | Integer[Array, ""],
    learning_rate: float | Float[Array, ""]
    ) -> LinearAssociativeMfc:
    "Initialize a linear associative feature-to-context memory"
    memory = jnp.eye(item_count, item_count + 2)
    memory = jnp.hstack([jnp.zeros((item_count, 1)), memory[:, :-1]])
    memory = memory * (1 - learning_rate)
    return LinearAssociativeMfc(memory)

@dispatch
def initialize(
    model_type: Val(LinearAssociativeMcf),
    item_count: int | Integer[Array, ""],
    shared_support: float | Float[Array, ""],
    item_support: float | Float[Array, ""]
    ) -> LinearAssociativeMcf:
    "Initialize a linear associative context-to-feature memory"
    memory = jnp.full((item_count, item_count), shared_support)
    memory = memory.at[jnp.diag_indices(item_count)].set(item_support)
    memory = jnp.vstack(
        (jnp.zeros((1, item_count)), memory, jnp.zeros((1, item_count))))
    return LinearAssociativeMcf(memory)

In [10]:
initialize(LinearAssociativeMfc, 10, .1)

/home/gunnj/jaxcmr/.venv/lib/python3.10/site-packages/plum/signature.py:203: UserWarning: Could not resolve the type hint of `plum.parametric.Val[__main__.LinearAssociativeMfc]()`. I have ended the resolution here to not make your code break, but some types might not be working correctly. Please open an issue at https://github.com/wesselb/plum.
  annotation = resolve_type_hint(p.annotation)
/home/gunnj/jaxcmr/.venv/lib/python3.10/site-packages/plum/type.py:262: UserWarning: Could not resolve the type hint of `plum.parametric.Val[__main__.LinearAssociativeMfc]()`. I have ended the resolution here to not make your code break, but some types might not be working correctly. Please open an issue at https://github.com/wesselb/plum.
  return _is_faithful(resolve_type_hint(x))
/home/gunnj/jaxcmr/.venv/lib/python3.10/site-packages/plum/type.py:262: UserWarning: Could not determine whether `plum.parametric.Val[__main__.LinearAssociativeMfc]()` is faithful or not. I have concluded that the type i

BeartypeDecorHintNonpepException: is_bearable() type hint plum.parametric.Val[__main__.LinearAssociativeMfc]() either PEP-noncompliant or currently unsupported by @beartype.